In [ ]:
import pandas as pd
import requests
import json
import os
from collections import defaultdict
import requests
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import pickle

# Function Definitions

In [ ]:
def get_short_name(full_query):
    return full_query.split("_unrelaxed", 1)[0]

def get_ptm(name, data, cell):
    query_s = get_short_name(cell)
    return data[name][query_s][2]

def extract_accession(target):
    """
    Extract the UniProt accession from the target string.
    Expected format: 'AF-A0A831PDK2-F1-model_v4'
    This function returns 'A0A831PDK2'.
    """
    try:
        # Split by '-' and take the second element (index 1)
        parts = target.split("-")
        accession = parts[1]  # e.g., 'A0A831PDK2'
        return accession
    except Exception as e:
        print(f"Error extracting accession from {target}: {e}")
        return None

def fetch_annotation(accession):
    """
    Fetch annotations from the UniProt REST API for the given accession.
    Returns a dictionary of selected annotation details.
    """
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.json"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            # Extract some useful annotations:
            protein_name = data.get("proteinDescription", {})\
                               .get("recommendedName", {})\
                               .get("fullName", {})\
                               .get("value")
            organism = data.get("organism", {}).get("scientificName")
            gene = None
            genes = data.get("genes", [])
            if genes:
                gene = genes[0].get("geneName", {}).get("value")
            # Return a dictionary of annotations.
            return {
                "uniprot_id": accession,
                "protein_name": protein_name,
                "organism": organism,
                "gene": gene,
            }
        else:
            print(f"Failed to fetch annotation for {accession}: HTTP {response.status_code}")
            return None
    except Exception as e:
        print(f"Error fetching annotation for {accession}: {e}")
        return None


def download_alphafold_pdb(uniprot_acc, version="v4", path="."):
    """
    Downloads the AlphaFold PDB file for a given UniProt accession and saves it to the specified directory.
    
    Args:
        uniprot_acc (str): The UniProt accession number.
        version (str): The model version (default "v4").
        path (str): Directory to save the file (default is current directory).
        
    Returns:
        str: Full path to the downloaded PDB file, or None if download fails.
    """
    # Construct the download URL
    url = f"https://alphafold.ebi.ac.uk/files/AF-{uniprot_acc}-F1-model_{version}.pdb"
    print(f"Attempting to download from: {url}")
    
    # Perform the HTTP GET request
    response = requests.get(url)
    
    if response.status_code == 200:
        # Make sure the output directory exists
        os.makedirs(path, exist_ok=True)
        
        filename = f"AF-{uniprot_acc}-F1-model_{version}.pdb"
        full_path = os.path.join(path, filename)
        
        with open(full_path, 'wb') as file:
            file.write(response.content)
        
        #print(f"Downloaded and saved as {full_path}")
        return full_path
    else:
        print(f"Download failed with status code {response.status_code}.")
        return None

def extract_avg_plddt(pdb_path):
    plddt_scores = []

    with open(pdb_path, 'r') as f:
        for line in f:
            if line.startswith("ATOM"):
                try:
                    b_factor = float(line[60:66].strip())
                    plddt_scores.append(b_factor)
                except ValueError:
                    continue  # In case parsing fails

    if plddt_scores:
        return round(sum(plddt_scores) / len(plddt_scores), 2)
    else:
        return None  # No ATOM entries found

def get_plddt(cell, scoredict):
    return scoredict[cell]


# (1) Foldseek 

In [ ]:
# Define the expected columns
foldseekcolumns = [
    "query", "target", "fident", "alnlen", "mismatch", "gapopen",
    "qstart", "qend", "tstart", "tend", "evalue", "bits",
    "qcov", "tcov", "alntmscore", "qtmscore", "ttmscore", "prob"
]

foldseekpath="/project3/data/results/foldseek/foldsearch3"
foldseekfiles = [
    "methanogen_mega.tsv",
    "methanogen_safe.tsv",
    "methanogen_unsafe.tsv",
    "methanotroph_mega.tsv",
    "methanotroph_safe.tsv",
    "methanotroph_unsafe.tsv"
]

methanogenfiles = [
    "methanogen_mega.tsv",
    "methanogen_safe.tsv",
    "methanogen_unsafe.tsv",
    "methanotroph_mega.tsv",
    "methanotroph_safe.tsv",
    "methanotroph_unsafe.tsv"
]

methanotrophfiles = [
    "methanotroph_mega.tsv",
    "methanotroph_safe.tsv",
    "methanotroph_unsafe.tsv"
]

In [ ]:
# 1. Load your JSON file
json_file = "colabfold_results_2.json"  # Adjust name/path
with open(json_file, "r") as f:
    data = json.load(f)

In [ ]:
# Dictionary to store each DataFrame
foldseek_dic = {}

# Read each TSV file and drop duplicate queries
for file in foldseekfiles:
    file_f = f"{foldseekpath}/{file}"
    df = pd.read_csv(file_f, sep="\t", names=foldseekcolumns)
    df["assembler"] = file.split("_")[1].split(".")[0]
    df.drop_duplicates(subset=["query"], inplace=True)
    foldseek_dic[file] = df


In [ ]:
for file in foldseekfiles:
    name = file.split(".tsv")[0]
    cache = {}  # dictionary to store computed values for each unique query
    
    # Define a function that uses the cache to avoid recalculating values
    def cached_get_ptm(query):
        if query not in cache:
            cache[query] = get_ptm(name, data, query)
        return cache[query]
    
    # Apply the cached function to all queries in the dataframe
    foldseek_dic[file]["qptm"] = foldseek_dic[file]["query"].apply(cached_get_ptm)


In [ ]:
# get plddt
def get_av_plddt(name, data, cell):
    # Assuming get_short_name is already defined.
    query_s = get_short_name(cell)
    pdb_path = data[name][query_s][1]
    return extract_avg_plddt(pdb_path)

for file in foldseekfiles:
    name = file.split(".tsv")[0]
    cache = {}  # dictionary to store computed values for each unique query
    
    # Define a function that uses the cache to avoid recalculating values
    def cached_get_avplddt(query):
        if query not in cache:
            cache[query] = get_av_plddt(name, data, query)
        return cache[query]
    
    # Apply the cached function to all queries in the dataframe
    foldseek_dic[file]["qplddt"] = foldseek_dic[file]["query"].apply(cached_get_avplddt)

In [ ]:
qptms = foldseek_dic[ "methanogen_unsafe.tsv"]["qptm"].values.tolist()

## Alphafold Model Scores

In [ ]:
pdboutpath="/project3/data/results/foldseek/pdbout"

In [ ]:
# for file in foldseekfiles:
#     targets = foldseek_dic[file]["target"].tolist()
#     for target in targets:
#         t = target.split("-")[1]
#         download_alphafold_pdb(t, "v4", pdboutpath)

In [ ]:
pdb_scores_dict = {}

for filename in os.listdir(pdboutpath):
    file = os.path.join(pdboutpath, filename)
    if file.endswith(".pdb"):
        accession = filename.split(".")[0]
        if accession not in pdb_scores_dict:
            pdb_scores_dict[accession] = extract_avg_plddt(file)
        else:
            print("already inside")

In [ ]:
for file in foldseekfiles:
    name = file.split(".tsv")[0]
    foldseek_dic[file]["tplddt"] = foldseek_dic[file]["target"].apply(lambda cell: get_plddt(cell,pdb_scores_dict))

# for file in foldseekfiles:
#     name = file.split(".tsv")[0]
#     cache = {}  # dictionary to store computed values for each unique target
    
#     # Define a function that uses the cache to avoid recalculating values
#     def cached_get_plddt(target):
#         if target not in cache:
#             cache[target] = get_plddt(target, pdb_scores_dict)
#         return cache[target]
    
#     # Apply the cached function to all targets in the dataframe
#     foldseek_dic[file]["tplddt"] = foldseek_dic[file]["target"].apply(cached_get_plddt)


## Plotting

In [ ]:
def plot_joint(df, fileinfo):
    df = df.copy()
    df["qtm_label"] = df["qtmscore"].apply(lambda x: "TM ≥ 0.8" if x >= 0.8 else "TM < 0.8")

    g = sns.jointplot(
        data=df,
        x='tplddt',
        y='qplddt',
        hue='qtm_label',
        kind='scatter',
        height=5,
        s=3,
        palette={"TM ≥ 0.8": "red", "TM < 0.8": "blue"}
    )

    fontSize=14

    # Set axis labels with fontsize 14
    g.ax_joint.set_xlabel("AlphaFold DB reference model av. pLDDT", fontsize=fontSize)
    g.ax_joint.set_ylabel("assembled protein model av. pLDDT", fontsize=fontSize)

    # Set ticks and tick label sizes to 14
    # g.ax_joint.set_xticks(np.arange(0, 120, 20))
    # g.ax_joint.set_yticks(np.arange(0, 120, 20))
    xticks = np.unique(np.append(np.arange(0, 120, 20), 90))
    yticks = np.unique(np.append(np.arange(0, 120, 20), 90))
    g.ax_joint.set_xticks(xticks)
    g.ax_joint.set_yticks(yticks)
    g.ax_joint.tick_params(axis='both', labelsize=fontSize)
    g.ax_marg_x.tick_params(axis='x', labelsize=fontSize)
    g.ax_marg_y.tick_params(axis='y', labelsize=fontSize)

    # Add threshold lines
    g.ax_joint.axhline(90, color='grey', linestyle='--', alpha=0.5)
    g.ax_joint.axvline(90, color='grey', linestyle='--', alpha=0.5)

    # Move the legend first (this creates the legend object we want to modify)
    sns.move_legend(g.ax_joint, title='TM score alignment', loc='best', markerscale=4)
    
    # Then retrieve and style it
    legend = g.ax_joint.get_legend()
    legend.set_title("TM score alignment", prop={"size": 12})
    for text in legend.get_texts():
        text.set_fontsize(12)

    plt.savefig(f'plotting-scripts/plots/model_accuracies_{fileinfo}.svg', format="svg", bbox_inches="tight")
    plt.show()


In [ ]:
def plot_query_ptm(df, fileinfo):
    # Create labeled column for hue based on qtmscore threshold
    df = df.copy()
    df["qtm_label"] = df["qtmscore"].apply(lambda x: "TM ≥ 0.8" if x >= 0.8 else "TM < 0.8")
    
    # Create the jointplot with query-only metrics:
    # x-axis: query pLDDT ('qplddt')
    # y-axis: query pTM score ('qptm')
    g = sns.jointplot(
        data=df,
        x='qplddt',
        y='qptm',
        hue='qtm_label',
        kind='scatter',
        height=5,
        s=3,
        palette={"TM ≥ 0.8": "red", "TM < 0.8": "blue"}
    )

    fontSize=14
    
    # Add threshold lines for reference:
    g.ax_joint.axhline(0.8, color='grey', linestyle='--', alpha=0.5)
    g.ax_joint.axvline(90, color='grey', linestyle='--', alpha=0.5)
    
    # Custom axis labels with fontsize=12
    g.set_axis_labels("assembled protein model av. pLDDT", "assembled protein model pTM score", fontsize=fontSize)
    
    # Set ticks: for x (pLDDT) from 0 to 110 and for y (pTM) from 0 to 1
    g.ax_joint.set_xticks(np.arange(0, 120, 20))
    g.ax_joint.set_yticks(np.linspace(0, 1, 11))
    
    # Set tick parameters for joint and marginal plots
    g.ax_joint.tick_params(axis='both', labelsize=fontSize)
    g.ax_marg_x.tick_params(axis='x', labelsize=fontSize)
    g.ax_marg_y.tick_params(axis='y', labelsize=fontSize)
    
    # Legend styling: set title and text font sizes, and enlarge markers
    sns.move_legend(g.ax_joint, title='TM score alignment', loc='best', markerscale=4)
    legend = g.ax_joint.get_legend()
    legend.set_title("TM score alignment", prop={"size": 12})
    for text in legend.get_texts():
        text.set_fontsize(12)
    
    plt.savefig(f'plotting-scripts/plots/query_ptm_plddt{fileinfo}.svg', format="svg", bbox_inches="tight")
    plt.show()


In [ ]:
def plot_fident_vs_qtmscore(df, fileinfo):
    """
    Create a density (hexbin) plot of fident (x-axis) versus qtmscore (y-axis).

    Parameters:
    - df: DataFrame that must contain 'fident' and 'qtmscore' columns.
    - fileinfo: A string used in the filename for saving the plot.
    """
    df = df.copy()
    
    # Create a smaller figure
    plt.figure(figsize=(6, 5))
    
    # Create a hexbin density plot
    hb = plt.hexbin(df['fident'], df['qtmscore'], gridsize=30, cmap='coolwarm', mincnt=1, edgecolors='none')
    cb = plt.colorbar(hb)
    cb.set_label('Counts', fontsize=14)
    
    # Set axis labels with font size 14
    plt.xlabel("Sequence Similarity", fontsize=14)
    plt.ylabel("TM Score", fontsize=14)
    
    # Adjust tick label sizes
    plt.tick_params(axis='both', labelsize=14)
    
    plt.grid(axis='y', linestyle='--')
    plt.tight_layout()
    
    plt.savefig(f'plotting-scripts/plots/fident_vs_qtmscore_{fileinfo}.svg', format="svg", bbox_inches="tight")
    plt.show()


## Clustering

In [ ]:
pdb_path = "/project3/data/results/foldseek/mergedpdbs/pool_id/foldseek_pdb/all.pdb"
cluster_tsv = "/project3/data/results/foldseek/mergedpdbs/pool_id/foldseek_cluster2/all_cluster.tsv"

In [ ]:
# Part 1: Read the TSV and save the unique set of values in column 1
df = pd.read_csv(cluster_tsv, sep="\t", header=None)
cluster_reps = set(df[0].astype(str).str.split("_").str[2].astype(int))

In [ ]:
pdb_file = pdb_path  # update with your PDB file path
model_title_dict = {}

with open(pdb_file, "r") as f:
    lines = f.readlines()

i = 0
while i < len(lines):
    line = lines[i]
    if line.startswith("MODEL"):
        try:
            model_num = int(line.strip().split()[1])
        except (IndexError, ValueError):
            print("Could not parse model number from line:", line)
            i += 1
            continue

        # Collect all TITLE lines following this MODEL
        title_parts = []
        j = i + 1
        while j < len(lines) and lines[j].startswith("TITLE"):
            parts = lines[j].strip().split(maxsplit=1)
            if len(parts) == 2:
                title_parts.append(parts[1][1:len(parts[1])])
            j += 1

        full_title = "".join(title_parts)
        model_title_dict[model_num] = full_title
        i = j  # jump to next block
    else:
        i += 1


In [ ]:
clust_rep_dict = {k: v for k, v in model_title_dict.items() if k in cluster_reps}
cleaned_values = set([val.split('_', 1)[1] for val in clust_rep_dict.values()])

In [ ]:
mgen_dfs = pd.concat([foldseek_dic[file] for file in methanogenfiles if file in foldseek_dic])
mgen_dfs.reset_index(drop=True, inplace=True)
mgen_dfs = mgen_dfs[mgen_dfs["query"].isin(cleaned_values)]

In [ ]:
plot_fident_vs_qtmscore(mgen_dfs, "methanogen_clust_cov90only_fident")

In [ ]:
plot_joint(mgen_dfs, "methanogen_clust_cov90only")

In [ ]:
with open('annotations.pkl', 'rb') as f:
    annots = pickle.load(f)

In [ ]:
methaneAnnot = list(set().union(*annots.values()))
methAnnot = [x.split("_", 1)[1] for x in methaneAnnot]

In [ ]:
results = []
# Loop over each key in the dictionary
for key, identifiers in annots.items():
    # Collect scores for each metric for the current key
    scores = {"fident": [], "qplddt": [], "tplddt": [], "qtmscore": []}
    
    for identifier in identifiers:
        # Split the identifier; here we take the second part as the new value
        new_value = identifier.split("_", 1)[1]
        # Filter mgen_dfs for rows where "query" matches new_value
        match = mgen_dfs[mgen_dfs["query"] == new_value]
        if not match.empty:
            scores["fident"].extend(match["fident"].tolist())
            scores["qplddt"].extend(match["qplddt"].tolist())
            scores["tplddt"].extend(match["tplddt"].tolist())
            scores["qtmscore"].extend(match["qtmscore"].tolist())
    
    # Calculate medians for each metric if scores exist
    median_fident = np.median(scores["fident"]) if scores["fident"] else None
    median_qLPPDT = np.median(scores["qplddt"]) if scores["qplddt"] else None
    median_tLPPDT = np.median(scores["tplddt"]) if scores["tplddt"] else None
    median_qTM = np.median(scores["qtmscore"]) if scores["qtmscore"] else None
    
    results.append({
        "Pathway": key,
        "Median similarity": round(median_fident, 2) if median_fident is not None else None,
        "Median assembly pLDDT": round(median_qLPPDT, 2) if median_qLPPDT is not None else None,
        "Median database pLDDT": round(median_tLPPDT, 2) if median_tLPPDT is not None else None,
        "Median TM score": round(median_qTM, 2) if median_qTM is not None else None
    })

# Create a DataFrame with the results
results_df = pd.DataFrame(results)

# Convert the DataFrame to a LaTeX table
latex_table = results_df.to_latex(index=False)
print(latex_table)
